In [ ]:
import sys

print (sys.executable)

# Optimisation et interprétation du modèle

## Contexte et objectif

Cette
étape
vise
à
optimiser
le
modèle
non
linéaire
retenu
à
l’étape
précédente, puis
à
interpréter
son
comportement
à
l’aide
de
méthodes
d’explicabilité
globale
et
locale.

Les
objectifs
sont
les
suivants:
- améliorer
les
performances
du
modèle
par
fine - tuning;
- comparer
plusieurs
méthodes
de
feature
importance
globale;
- analyser
l’effet
des
variables
sur
les
prédictions
individuelles.

Le
modèle
retenu
ici
est
une ** Random
Forest **, adaptée
à
un
problème
non
linéaire
et
compatible
avec ** TreeExplainer ** de
SHAP.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    precision_recall_curve,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.preprocessing import OneHotEncoder

sns.set_theme (style="whitegrid")
pd.set_option ("display.max_columns", 200)

## 1. Chargement des données préparées

Les données nettoyées et consolidées à l’étape précédente sont rechargées à partir du dossier `data/processed`.

La variable cible est ensuite standardisée puis encodée sous forme binaire afin de faciliter l’entraînement et l’évaluation du modèle.

In [ ]:
PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_central = pd.read_csv (DATA_PROCESSED / "df_central.csv")

df_central["a_quitte_l_entreprise"] = (
    df_central["a_quitte_l_entreprise"]
    .astype (str)
    .str.strip ( )
    .str.capitalize ( )
)

df_central["attrition_bin"] = df_central["a_quitte_l_entreprise"].apply (
    lambda x: 1 if x == "Oui" else 0
)

print ("PROJECT_ROOT :", PROJECT_ROOT)
print ("DATA_PROCESSED :", DATA_PROCESSED)
print ("df_central shape :", df_central.shape)
display (df_central.head ( ))

## 2. Fonctions utilitaires de préparation

Les fonctions ci-dessous permettent de structurer la préparation des données de manière plus lisible et réutilisable.

Elles prennent en charge :
- l’identification des variables numériques et catégorielles ;
- la construction de la cible ;
- la création de la matrice de features ;
- l’encodage des variables catégorielles ;
- la suppression des variables trop fortement corrélées.

In [ ]:
def split_features_by_type(df: pd.DataFrame):
    num_cols = df.select_dtypes (include=["number"]).columns.tolist ( )
    cat_cols = df.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )
    return num_cols, cat_cols


def build_target(df: pd.DataFrame, target_col: str = "attrition_bin") -> pd.Series:
    return df[target_col].copy ( )


def build_feature_matrix(
        df: pd.DataFrame,
        cols_to_drop: list[str] | None = None
) -> pd.DataFrame:
    if cols_to_drop is None:
        cols_to_drop = [
            "a_quitte_l_entreprise",
            "attrition_bin",
            "id_employee",
            "code_sondage",
            "eval_number",
        ]
    return df.drop (columns=[col for col in cols_to_drop if col in df.columns]).copy ( )


def fit_encoder_on_train(df: pd.DataFrame):
    df = df.copy ( )
    num_cols, cat_cols = split_features_by_type (df)

    if not cat_cols:
        return None, df

    encoder = OneHotEncoder (
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )

    encoded_array = encoder.fit_transform (df[cat_cols])

    encoded_df = pd.DataFrame (
        encoded_array,
        columns=encoder.get_feature_names_out (cat_cols),
        index=df.index
    )

    df_encoded = pd.concat ([df[num_cols], encoded_df], axis=1)
    return encoder, df_encoded


def transform_with_encoder(df: pd.DataFrame, encoder) -> pd.DataFrame:
    df = df.copy ( )
    num_cols, cat_cols = split_features_by_type (df)

    if encoder is None or not cat_cols:
        return df

    encoded_array = encoder.transform (df[cat_cols])

    encoded_df = pd.DataFrame (
        encoded_array,
        columns=encoder.get_feature_names_out (cat_cols),
        index=df.index
    )

    return pd.concat ([df[num_cols], encoded_df], axis=1)


def drop_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.8):
    corr_matrix = df.corr (numeric_only=True).abs ( )
    upper = corr_matrix.where (np.triu (np.ones (corr_matrix.shape), k=1).astype (bool))
    to_drop = [column for column in upper.columns if any (upper[column] > threshold)]
    return df.drop (columns=to_drop), to_drop

## 3. Feature engineering

Quelques
variables
dérivées
sont
construites
afin
d’enrichir
la
représentation
initiale
des
employés.

L’objectif
est
de
faire
émerger
des
signaux
potentiellement
utiles
à
la
classification, par
exemple
en
combinant:
- le
revenu
et
l’expérience;
- l’ancienneté
relative;
- la
charge
liée
à
la
distance
domicile - travail;
- un
score
synthétique
de
satisfaction.

In [ ]:
df_fe = df_central.copy ( )

df_fe["revenu_par_annee_experience"] = (
        df_fe["revenu_mensuel"] / (df_fe["annee_experience_totale"] + 1)
)

df_fe["part_anciennete_entreprise"] = (
        df_fe["annees_dans_l_entreprise"] / (df_fe["annee_experience_totale"] + 1)
)

df_fe["part_anciennete_poste"] = (
        df_fe["annees_dans_le_poste_actuel"] / (df_fe["annees_dans_l_entreprise"] + 1)
)

df_fe["charge_distance"] = (
        df_fe["distance_domicile_travail"] * df_fe["nombre_heures_travailless"]
)

satisfaction_cols = [
    "satisfaction_employee_environnement",
    "satisfaction_employee_nature_travail",
    "satisfaction_employee_equipe",
    "satisfaction_employee_equilibre_pro_perso",
]

existing_satisfaction_cols = [col for col in satisfaction_cols if col in df_fe.columns]

if existing_satisfaction_cols:
    df_fe["score_satisfaction_moyen"] = df_fe[existing_satisfaction_cols].mean (axis=1)

print ("df_fe shape :", df_fe.shape)
display (df_fe.head ( ))

## 4. Construction de la cible et des features

La
variable
cible
`attrition_bin`
est
isolée, puis
les
variables
explicatives
sont
regroupées
dans
une
matrice
de
features.

In [ ]:
y = build_target (df_fe)
X_raw = build_feature_matrix (df_fe)

print ("X_raw shape :", X_raw.shape)
print ("y shape :", y.shape)

## 5. Séparation train / test

Le jeu de test est conservé à part pour l’évaluation finale.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split (
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print ("X_train_raw :", X_train_raw.shape)
print ("X_test_raw  :", X_test_raw.shape)
print ("y_train     :", y_train.shape)
print ("y_test      :", y_test.shape)

print ("\nRépartition y_train :")
print (y_train.value_counts (normalize=True))

print ("\nRépartition y_test :")
print (y_test.value_counts (normalize=True))

## 6. Encodage des variables et préparation finale

Les
variables
catégorielles
sont
encodées
à
partir
du
seul
jeu
d’apprentissage
afin
d’éviter
toute
fuite
d’information.

Les
variables
trop
corrélées
sont
ensuite
retirées, puis
les
jeux
train
et
test
sont
réalignés
pour
garantir
la
même
structure
finale.

In [ ]:
encoder, X_train_prepared = fit_encoder_on_train (X_train_raw)
X_test_prepared = transform_with_encoder (X_test_raw, encoder)

X_train, dropped_corr_features = drop_highly_correlated_features (
    X_train_prepared,
    threshold=0.8
)

X_test = X_test_prepared.drop (columns=dropped_corr_features, errors="ignore")

X_train, X_test = X_train.align (
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print ("Shape final de X_train :", X_train.shape)
print ("Shape final de X_test  :", X_test.shape)
print ("NaN dans X_train :", X_train.isna ( ).sum ( ).sum ( ))
print ("NaN dans X_test  :", X_test.isna ( ).sum ( ).sum ( ))
print ("Nombre de variables supprimées :", len (dropped_corr_features))

## 7. Création d’un sous-jeu de validation

Le
jeu
de
test
reste
totalement
à
l’écart.

Le
jeu
d’apprentissage
est
séparé
en:
- un
sous - jeu
d’entraînement
pour
le
fine - tuning;
- un
sous - jeu
de
validation
pour
choisir
le
seuil
de
décision.

In [ ]:
X_subtrain, X_valid, y_subtrain, y_valid = train_test_split (
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

print ("X_subtrain :", X_subtrain.shape)
print ("X_valid    :", X_valid.shape)
print ("X_test     :", X_test.shape)

print ("\nRépartition y_subtrain :")
print (y_subtrain.value_counts (normalize=True))

print ("\nRépartition y_valid :")
print (y_valid.value_counts (normalize=True))

print ("\nRépartition y_test :")
print (y_test.value_counts (normalize=True))

## 8. Fine-tuning de la Random Forest

La
recherche
d’hyperparamètres
est
réalisée
par
validation
croisée
stratifiée
sur
le
sous - jeu
d’entraînement.

Le
score
retenu
est ** average
precision **, plus
pertinent
qu’une
simple
accuracy
dans
un
contexte
déséquilibré.

In [ ]:
rf_base = RandomForestClassifier (
    random_state=42,
    n_jobs=-1,
)

param_distributions = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [None, 6, 8, 10, 12, 16, 20],
    "min_samples_split": [2, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "class_weight": ["balanced", "balanced_subsample"],
    "bootstrap": [True],
}

cv = StratifiedKFold (
    n_splits=5,
    shuffle=True,
    random_state=42,
)

random_search = RandomizedSearchCV (
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42,
    refit=True,
)

random_search.fit (X_subtrain, y_subtrain)

print ("Meilleurs paramètres :")
print (random_search.best_params_)

print ("\nMeilleur score CV (average precision) :")
print (round (random_search.best_score_, 4))

## 6. Feature importance globale — Permutation Importance

La permutation importance mesure la baisse de performance provoquée par la perturbation d’une variable.

Elle est souvent plus fidèle au comportement réel du modèle que l’importance native.

In [ ]:
best_rf_final = RandomForestClassifier (
    **random_search.best_params_,
    random_state=42,
    n_jobs=-1,
)

best_rf_final.fit (X_train, y_train)

print (best_rf_final)

In [ ]:
perm = permutation_importance (
    best_rf_final,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
    scoring="average_precision",
)

permutation_imp = (
    pd.Series (perm.importances_mean, index=X_test.columns)
    .sort_values (ascending=False)
)

display (permutation_imp.head (20))

In [ ]:
cv_results = pd.DataFrame (random_search.cv_results_).sort_values ("rank_test_score")

display (
    cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "param_n_estimators",
            "param_max_depth",
            "param_min_samples_split",
            "param_min_samples_leaf",
            "param_max_features",
            "param_class_weight",
        ]
    ].head (10)
)

## 9. Choix du seuil de décision sur validation

Le meilleur modèle issu du fine-tuning est entraîné sur le sous-jeu d’entraînement.

Le seuil de décision est ensuite choisi sur le jeu de validation à partir de la courbe précision-rappel, sans utiliser le jeu de test.

In [ ]:
best_rf = random_search.best_estimator_
best_rf.fit (X_subtrain, y_subtrain)

y_valid_scores = best_rf.predict_proba (X_valid)[:, 1]

precision_valid, recall_valid, thresholds_valid = precision_recall_curve (y_valid, y_valid_scores)

if len (thresholds_valid) > 0:
    f1_scores_valid = (
            2 * precision_valid[:-1] * recall_valid[:-1]
            / (precision_valid[:-1] + recall_valid[:-1] + 1e-9)
    )
    best_idx = np.nanargmax (f1_scores_valid)
    best_threshold = float (thresholds_valid[best_idx])
else:
    best_threshold = 0.5
    f1_scores_valid = np.array ([])

print ("Seuil optimal choisi sur validation :", round (best_threshold, 3))

In [ ]:
threshold_df = pd.DataFrame ({
    "threshold": thresholds_valid,
    "precision": precision_valid[:-1],
    "recall": recall_valid[:-1],
    "f1": f1_scores_valid,
})

display (threshold_df.sort_values ("f1", ascending=False).head (10))

## 10. Réentraînement final et évaluation sur test

Le
modèle
final
est
réentraîné
sur
l’ensemble
du
jeu
d’apprentissage
avec
les
meilleurs
hyperparamètres.

Le
seuil
retenu
sur
validation
est
ensuite
appliqué
au
jeu
de
test.

In [ ]:
best_rf_final = RandomForestClassifier (
    **random_search.best_params_,
    random_state=42,
    n_jobs=-1,
)

best_rf_final.fit (X_train, y_train)

y_train_scores = best_rf_final.predict_proba (X_train)[:, 1]
y_test_scores = best_rf_final.predict_proba (X_test)[:, 1]

y_train_pred_default = (y_train_scores >= 0.5).astype (int)
y_test_pred_default = (y_test_scores >= 0.5).astype (int)

y_train_pred_threshold = (y_train_scores >= best_threshold).astype (int)
y_test_pred_threshold = (y_test_scores >= best_threshold).astype (int)

results_final = pd.DataFrame([
    {
        "configuration": "train_seuil_0.5",
        "accuracy": accuracy_score(y_train, y_train_pred_default),
        "precision": precision_score(y_train, y_train_pred_default, zero_division=0),
        "recall": recall_score(y_train, y_train_pred_default, zero_division=0),
        "f1": f1_score(y_train, y_train_pred_default, zero_division=0),
        "average_precision": average_precision_score(y_train, y_train_scores),
    },
    {
        "configuration": "test_seuil_0.5",
        "accuracy": accuracy_score(y_test, y_test_pred_default),
        "precision": precision_score(y_test, y_test_pred_default, zero_division=0),
        "recall": recall_score(y_test, y_test_pred_default, zero_division=0),
        "f1": f1_score(y_test, y_test_pred_default, zero_division=0),
        "average_precision": average_precision_score(y_test, y_test_scores),
    },
    {
        "configuration": f"train_seuil_{best_threshold:.3f}",
        "accuracy": accuracy_score(y_train, y_train_pred_threshold),
        "precision": precision_score(y_train, y_train_pred_threshold, zero_division=0),
        "recall": recall_score(y_train, y_train_pred_threshold, zero_division=0),
        "f1": f1_score(y_train, y_train_pred_threshold, zero_division=0),
        "average_precision": average_precision_score(y_train, y_train_scores),
    },
    {
        "configuration": f"test_seuil_{best_threshold:.3f}",
        "accuracy": accuracy_score(y_test, y_test_pred_threshold),
        "precision": precision_score(y_test, y_test_pred_threshold, zero_division=0),
        "recall": recall_score(y_test, y_test_pred_threshold, zero_division=0),
        "f1": f1_score(y_test, y_test_pred_threshold, zero_division=0),
        "average_precision": average_precision_score(y_test, y_test_scores),
    },
]).round(3)

display(results_final)

In [ ]:
print ("TEST - seuil 0.5")
print (classification_report (y_test, y_test_pred_default, zero_division=0))

print ("\nTEST - seuil optimisé")
print (classification_report (y_test, y_test_pred_threshold, zero_division=0))

In [ ]:
fig, axes = plt.subplots (1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions (
    y_test,
    y_test_pred_default,
    ax=axes[0],
    cmap="Blues",
    normalize="true",
    values_format=".2f",
)
axes[0].set_title ("Test - seuil 0.5")

ConfusionMatrixDisplay.from_predictions (
    y_test,
    y_test_pred_threshold,
    ax=axes[1],
    cmap="Blues",
    normalize="true",
    values_format=".2f",
)
axes[1].set_title (f"Test - seuil optimisé ({best_threshold:.3f})")

plt.tight_layout ( )
plt.show ( )

## 11. Feature importance globale — importance native de la Random Forest

Cette première mesure donne une importance moyenne issue directement du modèle.

Elle est rapide à calculer, mais peut être biaisée en présence de variables corrélées ou de variables à forte cardinalité.

In [ ]:
native_importance = (
    pd.Series (best_rf_final.feature_importances_, index=X_train.columns)
    .sort_values (ascending=False)
)

display (native_importance.head (20))

plt.figure(figsize=(8, 6))
native_importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 - importance native de la Random Forest")
plt.xlabel("Importance")
plt.ylabel("Variable")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
native_importance = (
    pd.Series (best_rf_final.feature_importances_, index=X_train.columns)
    .sort_values (ascending=False)
)

display (native_importance.head (20))

In [ ]:
plt.figure (figsize=(8, 6))
native_importance.head (15).sort_values ( ).plot (kind="barh")
plt.title ("Top 15 - importance native de la Random Forest")
plt.xlabel ("Importance")
plt.ylabel ("Variable")
plt.grid (axis="x", alpha=0.3)
plt.tight_layout ( )
plt.show ( )

## 12. Feature importance globale — Permutation Importance

La
permutation
importance
mesure
la
baisse
de
performance
provoquée
par
la
perturbation
d’une
variable.

Elle
est
souvent
plus
fidèle
au
comportement
réel
du
modèle
que
l’importance
native.

In [ ]:
perm = permutation_importance (
    best_rf_final,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
    scoring="average_precision",
)

permutation_imp = (
    pd.Series (perm.importances_mean, index=X_test.columns)
    .sort_values (ascending=False)
)

display (permutation_imp.head (20))

In [ ]:
plt.figure (figsize=(8, 6))
permutation_imp.head (15).sort_values ( ).plot (kind="barh")
plt.title ("Top 15 - permutation importance")
plt.xlabel ("Baisse moyenne du score")
plt.ylabel ("Variable")
plt.grid (axis="x", alpha=0.3)
plt.tight_layout ( )
plt.show ( )

## 11. Interprétation des résultats

Le fine-tuning a permis d’obtenir une version optimisée de la Random Forest, mieux adaptée au problème d’attrition.

L’analyse des importances globales montre plusieurs points de convergence entre :
- l’importance native du modèle ;
- la permutation importance ;
- les valeurs SHAP moyennes absolues.

Lorsque certaines variables apparaissent importantes dans les trois méthodes, cela renforce la confiance dans leur rôle global dans les prédictions. À l’inverse, des divergences peuvent apparaître pour des variables corrélées ou pour des variables dont l’effet dépend d’interactions complexes.

Les graphiques SHAP permettent d’aller plus loin que le simple classement des variables. Le beeswarm plot montre non seulement quelles variables influencent le plus le modèle, mais aussi dans quel sens elles poussent la prédiction. Une valeur SHAP positive contribue à augmenter la probabilité de départ, tandis qu’une valeur SHAP négative contribue à la diminuer.

Enfin, les waterfall plots permettent une lecture locale de la prédiction. Ils montrent, pour un individu donné, quelles variables ont le plus contribué à pousser la prédiction vers l’attrition ou vers le maintien dans l’entreprise. Cette lecture locale est particulièrement utile pour expliquer concrètement le comportement du modèle sur des cas individuels.

In [ ]:
def get_positive_class_shap_explanation(explainer, X: pd.DataFrame):
    explanation = explainer (X)

    values = explanation.values
    base_values = explanation.base_values

    if values.ndim == 3:
        values = values[:, :, 1]

        if np.ndim (base_values) == 2:
            base_values = base_values[:, 1]
        elif np.ndim (base_values) == 1 and len (base_values) == 2:
            base_values = np.repeat (base_values[1], X.shape[0])

    return shap.Explanation (
        values=values,
        base_values=base_values,
        data=X.values,
        feature_names=X.columns.tolist ( ),
    )

In [ ]:
X_shap = X_train.sample (min (300, len (X_train)), random_state=42)

tree_explainer = shap.TreeExplainer (best_rf_final)
shap_explanation = get_positive_class_shap_explanation (tree_explainer, X_shap)

In [ ]:
shap_global_importance = (
    pd.Series (np.abs (shap_explanation.values).mean (axis=0), index=X_shap.columns)
    .sort_values (ascending=False)
)

display (shap_global_importance.head (20))

In [ ]:
shap.plots.beeswarm (shap_explanation, max_display=15)

## 14. Comparaison des méthodes de feature importance globale

Les trois approches suivantes sont comparées :
- importance native ;
- permutation importance ;
- SHAP (moyenne des valeurs absolues).

In [ ]:
importance_compare = pd.concat (
    [
        native_importance.rename ("native_importance"),
        permutation_imp.rename ("permutation_importance"),
        shap_global_importance.rename ("shap_mean_abs"),
    ],
    axis=1,
).fillna (0)

display (
    importance_compare.sort_values ("shap_mean_abs", ascending=False).head (20)
)

In [ ]:
top_features_compare = importance_compare.sort_values (
    "shap_mean_abs", ascending=False
).head (10)

top_features_compare = top_features_compare.sort_values ("shap_mean_abs")

ax = top_features_compare.plot (kind="barh", figsize=(10, 6))
plt.title ("Comparaison des importances globales")
plt.xlabel ("Importance")
plt.ylabel ("Variable")
plt.grid (axis="x", alpha=0.3)
plt.tight_layout ( )
plt.show ( )

## 15. Lecture du sens des Shapley values

Une valeur SHAP positive pousse la prédiction vers la classe positive (`attrition = 1`), tandis qu’une valeur négative la pousse vers la classe négative.

Le graphique suivant aide à interpréter le sens d’une variable et ses interactions avec une autre.

In [ ]:
top_shap_features = shap_global_importance.index.tolist ( )

feature_1 = top_shap_features[0]
feature_2 = top_shap_features[1] if len (top_shap_features) > 1 else top_shap_features[0]

print ("Variable principale :", feature_1)
print ("Variable de coloration :", feature_2)

shap.plots.scatter (
    shap_explanation[:, feature_1],
    color=shap_explanation[:, feature_2],
)

## 16. Explications locales avec Waterfall Plot

Les graphiques waterfall permettent d’expliquer une prédiction individuelle.

On sélectionne ici :
- un employé ayant quitté l’entreprise ;
- un employé resté dans l’entreprise.

In [ ]:
X_test_reset = X_test.reset_index (drop=True)
y_test_reset = y_test.reset_index (drop=True)

test_results = pd.DataFrame ({
    "y_true": y_test_reset,
    "proba_attrition": best_rf_final.predict_proba (X_test_reset)[:, 1],
})

display (test_results.head ( ))

In [ ]:
idx_positive = test_results.loc[test_results["y_true"] == 1, "proba_attrition"].idxmax ( )
idx_negative = test_results.loc[test_results["y_true"] == 0, "proba_attrition"].idxmin ( )

print ("Index exemple classe positive :", idx_positive)
print ("Index exemple classe négative :", idx_negative)

In [ ]:
shap_test_explanation = get_positive_class_shap_explanation (tree_explainer, X_test_reset)

In [ ]:
print ("Exemple classe positive")
display (X_test_reset.loc[[idx_positive]])
print ("Probabilité de départ :", round (test_results.loc[idx_positive, "proba_attrition"], 3))

shap.plots.waterfall (shap_test_explanation[idx_positive], max_display=12)

In [ ]:
print ("Exemple classe négative")
display (X_test_reset.loc[[idx_negative]])
print ("Probabilité de départ :", round (test_results.loc[idx_negative, "proba_attrition"], 3))

shap.plots.waterfall (shap_test_explanation[idx_negative], max_display=12)

## 17. Analyse d’un faux négatif

L’analyse d’un faux négatif peut être particulièrement utile dans un contexte métier RH, car elle permet de comprendre pourquoi un employé ayant réellement quitté l’entreprise n’a pas été détecté par le modèle.

In [ ]:
test_results["y_pred_threshold"] = y_test_pred_threshold

false_negatives = test_results[
    (test_results["y_true"] == 1) & (test_results["y_pred_threshold"] == 0)
    ]

if len (false_negatives) > 0:
    idx_fn = false_negatives["proba_attrition"].idxmax ( )
    print ("Index faux négatif :", idx_fn)
    print ("Probabilité de départ :", round (test_results.loc[idx_fn, "proba_attrition"], 3))
    shap.plots.waterfall (shap_test_explanation[idx_fn], max_display=12)
else:
    print ("Aucun faux négatif avec le seuil retenu.")

## 18. Interprétation des résultats

Le fine-tuning a permis d’obtenir une version optimisée de la Random Forest, mieux adaptée au problème d’attrition.

L’analyse des importances globales montre plusieurs points de convergence entre :
- l’importance native du modèle ;
- la permutation importance ;
- les valeurs SHAP moyennes absolues.

Lorsque certaines variables apparaissent importantes dans les trois méthodes, cela renforce la confiance dans leur rôle global dans les prédictions. À l’inverse, des divergences peuvent apparaître pour des variables corrélées ou pour des variables dont l’effet dépend d’interactions complexes.

Les graphiques SHAP permettent d’aller plus loin que le simple classement des variables. Le beeswarm plot montre non seulement quelles variables influencent le plus le modèle, mais aussi dans quel sens elles poussent la prédiction. Une valeur SHAP positive contribue à augmenter la probabilité de départ, tandis qu’une valeur SHAP négative contribue à la diminuer.

Enfin, les waterfall plots permettent une lecture locale de la prédiction. Ils montrent, pour un individu donné, quelles variables ont le plus contribué à pousser la prédiction vers l’attrition ou vers le maintien dans l’entreprise. Cette lecture locale est particulièrement utile pour expliquer concrètement le comportement du modèle sur des cas individuels.

## Conclusion

Cette étape a permis :
- d’optimiser le modèle par recherche d’hyperparamètres ;
- de choisir le seuil de décision sur validation plutôt que sur test ;
- de comparer plusieurs approches d’importance globale ;
- d’expliquer localement certaines prédictions grâce aux waterfall plots.

Le modèle final reste perfectible, mais il est désormais mieux calibré et surtout mieux interprétable, ce qui renforce sa valeur dans un contexte RH.